## Primera Llamada al Modelo (con OpenRouter)

En este ejercicio, aprenderemos a realizar nuestra primera llamada a un modelo de lenguaje usando **OpenRouter API**.

# 1. OpenRouter API - Conexión Directa con OpenAI Client

## Objetivos de Aprendizaje
- Configurar una conexión directa con OpenRouter usando el cliente OpenAI
- Comprender los parámetros básicos de configuración de API
- Implementar llamadas básicas a modelos de lenguaje
- Aplicar mejores prácticas de seguridad con API keys

## Introducción
OpenRouter proporciona acceso a múltiples modelos de lenguaje mediante una API compatible con OpenAI. En este notebook aprenderemos a:
1. Configurar el entorno y las credenciales
2. Establecer una conexión con la API
3. Realizar llamadas básicas al modelo
4. Explorar diferentes parámetros de configuración
5. Experimentar con modelos gratuitos

## Configuración de Variables de Entorno

Antes de ejecutar el código, asegúrate de tener configurada tu clave de OpenRouter:

**Opción 1: Variables de entorno**
```bash
export OPENROUTER_API_KEY="sk-or-v1-..."
```

**Opción 2: Archivo .env**
Crea un archivo `.env` en el mismo directorio con:
```
OPENROUTER_API_KEY=sk-or-v1-...
```

**Opción 3: Ingreso interactivo** (el notebook te lo pedirá si no encuentra la clave).

**Mejores Prácticas de Seguridad:**
- Nunca hardcodees API keys en el código
- Usa variables de entorno o archivos .env
- No compartas credenciales en repositorios públicos
- Rota las API keys regularmente

## Instalación de Dependencias
```bash
pip install openai python-dotenv
```

In [ ]:
# Instalación de dependencias (ejecutar una sola vez)
%pip install openai python-dotenv -q
print("✅ Dependencias instaladas")

In [ ]:
# Importar las bibliotecas necesarias
import os
from openai import OpenAI
from dotenv import load_dotenv
from getpass import getpass

# Verificar versiones
print("OpenAI library version:", __import__('openai').__version__)
print("Python version:", __import__('sys').version)
print()

# Cargar variables desde .env (si existe)
load_dotenv()

# --- Configuración de OpenRouter ---
BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY")

# Si no está en el entorno, pedirla interactivamente
if not API_KEY:
    print("🔑 No se encontró OPENROUTER_API_KEY en el entorno.")
    API_KEY = getpass("Ingresa tu clave de OpenRouter (sk-or-v1-...): ")
    os.environ["OPENROUTER_API_KEY"] = API_KEY

# Validar que la clave no esté vacía
if not API_KEY or len(API_KEY) < 20:
    raise ValueError("❌ La clave parece incorrecta. Debe empezar por 'sk-or-v1-' y tener al menos 20 caracteres.")

# Creamos el cliente
client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
)

# (Opcional) Identificador de tu app para OpenRouter
DEFAULT_HEADERS = {
    "HTTP-Referer": "http://localhost:8000",  # Cambia por tu dominio
    "X-Title": "Primera Llamada OpenRouter",
}

print("\n✅ Cliente OpenRouter configurado correctamente.")
print(f"   Base URL: {BASE_URL}")
print(f"   API Key presente: {'✅ Sí' if API_KEY else '❌ No'}")
print(f"   Clave (primeros 10 caracteres): {API_KEY[:10]}...")

# --- Modelo por defecto (gratuito) ---
MODELO_POR_DEFECTO = "minimax/minimax-m3:free"  # Modelo gratuito disponible
print(f"   Modelo por defecto: {MODELO_POR_DEFECTO}")

In [ ]:
# Primera llamada básica al modelo (parametrizada)
def llamada_basica(modelo=MODELO_POR_DEFECTO):
    """
    Realiza una llamada básica al modelo con un prompt simple.
    
    Args:
        modelo (str): Identificador del modelo en OpenRouter.
    """
    try:
        response = client.chat.completions.create(
            model=modelo,
            messages=[
                {"role": "user", "content": "Hola, ¿cómo estás? Responde en una oración."}
            ],
            temperature=0.1,
            max_tokens=150,
            extra_headers=DEFAULT_HEADERS  # ¡Importante! Identificar tu app
        )
        
        print("=== Respuesta del Modelo ===")
        print(response.choices[0].message.content)
        print("\n=== Información Técnica ===")
        print(f"Modelo usado: {response.model}")
        print(f"Tokens totales: {response.usage.total_tokens}")
        print(f"  - Entrada: {response.usage.prompt_tokens}")
        print(f"  - Salida: {response.usage.completion_tokens}")
        
    except Exception as e:
        print(f"❌ Error en la llamada: {e}")
        print("Verifica tu configuración y conexión a internet")

# Ejecutar la función
print(f"✅ Usando modelo: {MODELO_POR_DEFECTO}\n")
llamada_basica()

## Usando Roles del Sistema

El rol "system" permite establecer el comportamiento y contexto del asistente antes de la conversación.

In [ ]:
# Ejemplo con mensaje de sistema
def usar_mensaje_sistema(modelo=MODELO_POR_DEFECTO):
    try:
        response = client.chat.completions.create(
            model=modelo,
            messages=[
                {
                    "role": "system", 
                    "content": "Eres un experto en tecnología que explica conceptos complejos de manera simple y amigable. Siempre incluyes ejemplos prácticos."
                },
                {
                    "role": "user", 
                    "content": "¿Qué es una API?"
                }
            ],
            temperature=0.7,
            max_tokens=250,  # Aumentado para respuestas más completas
            extra_headers=DEFAULT_HEADERS
        )
        
        print("=== Respuesta con Mensaje de Sistema ===")
        print(response.choices[0].message.content)
        print("\n=== Información Técnica ===")
        print(f"Modelo: {response.model}")
        print(f"Tokens totales: {response.usage.total_tokens}")
        
    except Exception as e:
        print(f"❌ Error: {e}")

# Ejecutar función
usar_mensaje_sistema()

## Explorando Parámetros de Configuración

Los parámetros más importantes al hacer llamadas a LLMs son:

- **temperature**: Controla la creatividad (0.0 = determinístico, 1.0 = muy creativo)
- **max_tokens**: Límite de tokens en la respuesta
- **model**: El modelo específico a usar (formato `publisher/modelo` en OpenRouter)
- **messages**: Array de mensajes con roles (system, user, assistant)

In [ ]:
# Comparando diferentes valores de temperature
def comparar_temperature(modelo=MODELO_POR_DEFECTO):
    prompt = "Escribe una historia muy corta sobre un robot que aprende a cocinar."
    temperatures = [0.1, 0.5, 0.9]
    
    for temp in temperatures:
        print(f"\n{'='*50}")
        print(f"TEMPERATURE: {temp}")
        print('='*50)
        
        try:
            response = client.chat.completions.create(
                model=modelo,
                messages=[{"role": "user", "content": prompt}],
                temperature=temp,
                max_tokens=250,  # Aumentado para historias completas
                extra_headers=DEFAULT_HEADERS
            )
            
            print(response.choices[0].message.content)
            print(f"\n📊 Tokens usados: {response.usage.total_tokens}")
            
        except Exception as e:
            print(f"❌ Error: {e}")

# Ejecutar comparación
comparar_temperature()

## Explorando Modelos Gratuitos

OpenRouter ofrece varios modelos gratuitos (con sufijo `:free`). Aquí probamos algunos disponibles.

In [ ]:
# Función para probar diferentes modelos gratuitos
def probar_modelo(model_id, prompt="Dime un dato curioso sobre inteligencia artificial"):
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5,
            max_tokens=150,
            extra_headers=DEFAULT_HEADERS
        )
        print(f"\n🔹 Modelo: {model_id}")
        print(f"   Respuesta: {response.choices[0].message.content}")
        print(f"   Tokens: {response.usage.total_tokens}")
        return response
    except Exception as e:
        print(f"❌ Error con {model_id}: {e}")
        return None

# Lista de modelos gratuitos verificados
modelos_gratuitos = [
    "minimax/minimax-m3:free",
    "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "cohere/north-mini-code:free",
]

print("=== PROBANDO MODELOS GRATUITOS ===")
for modelo in modelos_gratuitos:
    probar_modelo(modelo)

print("\n💡 Para ver todos los modelos gratuitos disponibles, visita:")
print("   https://openrouter.ai/chat")

## Obtención Dinámica de Modelos Gratuitos

OpenRouter permite consultar la lista de modelos disponibles. Aquí obtenemos los que son gratuitos.

In [ ]:
import requests

def obtener_modelos_gratuitos(api_key):
    """
    Consulta la API de OpenRouter y devuelve los IDs de los modelos gratuitos disponibles.
    """
    url = "https://openrouter.ai/api/v1/models"
    headers = {"Authorization": f"Bearer {api_key}"}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
        # Filtrar modelos cuyo ID termina en ':free'
        modelos_gratis = [m['id'] for m in data['data'] if m['id'].endswith(':free')]
        return modelos_gratis
    except Exception as e:
        print(f"❌ Error al obtener modelos: {e}")
        return []

# Obtener y mostrar los primeros 5 modelos gratuitos
modelos_disponibles = obtener_modelos_gratuitos(API_KEY)
print(f"✅ Modelos gratuitos disponibles: {len(modelos_disponibles)}")
if modelos_disponibles:
    print("\n🔍 Primeros 5 modelos gratuitos:")
    for i, m in enumerate(modelos_disponibles[:5], 1):
        print(f"   {i}. {m}")
    
    # Probar el primero de la lista
    if modelos_disponibles:
        print("\n--- Probando el primer modelo gratuito ---")
        probar_modelo(modelos_disponibles[0])
else:
    print("⚠️ No se encontraron modelos gratuitos o hubo un error.")

## Ejercicios Prácticos

### Ejercicio 1: Experimentar con Diferentes Modelos
Modifica el código para probar diferentes modelos disponibles:
- `minimax/minimax-m3:free` (gratuito)
- `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free` (gratuito)
- `cohere/north-mini-code:free` (gratuito)

### Ejercicio 2: Crear un Asistente Especializado
Diseña un mensaje de sistema para crear un asistente especializado en un tema específico (ejemplo: finanzas, salud, educación).

### Ejercicio 3: Optimización de Tokens
Experimenta con diferentes valores de `max_tokens` para encontrar el equilibrio entre respuesta completa y eficiencia de costos.

### Ejercicio 4 (Avanzado): Modelos de Pago
Si tienes créditos en OpenRouter, prueba modelos como `openai/gpt-4o` o `anthropic/claude-3.5-sonnet`.

## Conceptos Clave

1. **Configuración segura** de APIs usando variables de entorno o archivos `.env`.
2. **Parámetros básicos** para controlar el comportamiento del modelo (temperature, max_tokens).
3. **Manejo de errores** en llamadas a APIs.
4. **Roles de mensajes** (system, user, assistant).
5. **Monitoreo de uso** de tokens y costos.
6. **Modelos gratuitos** en OpenRouter (sufijo `:free`).

## Próximos Pasos

En el siguiente notebook exploraremos cómo LangChain simplifica y abstrae estas operaciones, proporcionando herramientas más poderosas para el desarrollo de aplicaciones con LLMs.